In [1]:
import os
import sys
p = os.path.abspath('../..')
if p not in sys.path:
    sys.path.append(p)

from numcodecs import Blosc
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import glob
import json
import zarr
import waveorder as wo
from recOrder.recOrder.compute.QLIPP_compute import reconstruct_QLIPP_3D, initialize_reconstructor, load_bg
from recOrder.recOrder.io.reader import MicromanagerReader

SyntaxError: invalid syntax (QLIPP_compute.py, line 38)

In [2]:
def gather_data_paths(data_folder, sample_folders):
    dataPaths = []
    for j in sample_folders:
        temp = []
        temp_bg = []
        fold = os.path.join(data_folder,j)
        for dirs, folders, filenames in os.walk(fold):
            filenames.sort()

            for filename in filenames:

                ## Do not account for hidden files

                if "test" in filename:
                    continue

                if "Test" in filename:
                    continue

                if filename.endswith('.tif'):
                    temp.append(os.path.join(fold, os.path.join(dirs,filename)))

                else:
                    continue

        dataPaths.append(np.array(temp))
            
    data_paths_final = []
    for sample in dataPaths:
        temp_bg = []
        temp_C1 = []
        temp_C2 = []
        for i in sample:
            if 'old' in i:
                continue
            if 'BG' in i:
                temp_bg.append(i)
            if 'Coverslip_1' in i:
                temp_C1.append(i)
            if 'Coverslip_2'in i:
                temp_C2.append(i)

        data_paths_final.append([temp_bg,temp_C1,temp_C2])
        
    return data_paths_final

def init_zarr_store(data_path):
    chunk_size = (1, 65, 2048, 2448) # Dimension order PZYX
    compressor=Blosc(cname='zstd', clevel=1, shuffle=Blosc.SHUFFLE)
    
    data_shape = (10, 65, 2048, 2448)
    
    if not data_path.endswith('.zarr'):
        data_path += '.zarr'
    
    if not os.path.exists(data_path):
        os.makedirs(data_path)
        
    zarr_store = zarr.open(data_path, mode='a')
    
    
    datasets = ['Phase3D', 'Retardance', 'Orientation', 'BF']
    groups = ['RSV48', 'RSV24', 'Mock48', 'Mock24']
    
    for group in groups:
        zarr_store.create_group(group)
        for ds in datasets:
            zarr_store[group].zeros(ds, shape=data_shape, chunks=chunk_size, dtype='float32',
                                        compressor=compressor)
            
    return zarr_store

## Gather Data Paths

In [3]:
slideA = zarr.open('/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_18_40x_04NA_A549/SlideA/raw_data.zarr')

In [10]:
slideA['Reshape_test'].info

Name,/Reshape_test
Type,zarr.core.Array
Data type,uint16
Shape,"(40, 6, 65, 2048, 2448)"
Chunk shape,"(5, 1, 9, 256, 612)"
Order,C
Read-only,False
Compressor,"Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0)"
Store type,zarr.storage.DirectoryStore
No. bytes,156421324800 (145.7G)
No. bytes stored,144199300473 (134.3G)


In [5]:
data_folder = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_25_40X_04NA_A549/'

sample_paths = glob.glob(data_folder+'*')
sample_paths.pop(-1)

'/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_25_40X_04NA_A549/Calibration'

In [24]:
sample_paths[6][67:]

'MOCK_IFNL_24'

In [10]:
data = MicromanagerReader(sample_paths[0]+'/_1', extract_data=True)

[INFO: reader: 127 2021-03-05 09:26:58,593] checking DisplaySettings.json for ome-master records
[INFO: reader: 127 2021-03-05 09:26:58,593] checking _1_MMStack_7.ome.tif for ome-master records
[WARNING: tifffile:15794 2021-03-05 09:26:58,719] read_json: invalid JSON
[INFO: reader: 101 2021-03-05 09:26:58,721] extracting data from /gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_25_40X_04NA_A549/MOCK_IFNA_48/_1/_1_MMStack_7.ome.tif
[WARNING: tifffile:15794 2021-03-05 09:27:13,520] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,521] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,521] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,522] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,522] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,522] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,523] OME series: index out of ra

[WARNING: tifffile:15794 2021-03-05 09:27:13,562] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,564] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,564] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,565] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,565] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,565] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,566] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,566] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,567] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,568] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,568] OME series: index out of range
[WARNING: tifffile:15794 2021-03-05 09:27:13,569] OME series: index out of range
[WARNING: tifffile:15794 202

In [11]:
data._positions

{0: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 1: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 2: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 3: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 4: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 5: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 6: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 7: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 8: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 9: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>}

In [5]:
def pos_list_parser(path):
    poslist = json.load(open(path,'r'))
    for line in range(len(poslist['map']['StagePositions']['array'])):
        print(poslist['map']['StagePositions']['array'][line]['Label']['scalar'])

In [6]:
pos_list_parser('/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_18_40x_04NA_A549/SlideA/pos.pos')

RSV48_1
RSV48_2
RSV48_3
RSV48_4
RSV48_5
RSV48_6
RSV48_7
RSV48_8
RSV48_9
RSV48_10
RSV24_1
RSV24_2
RSV24_3
RSV24_4
RSV24_5
RSV24_6
RSV24_7
RSV24_8
RSV24_9
RSV24_10
MOCK48_1
MOCK48_2
MOCK48_3
MOCK48_4
MOCK48_5
MOCK48_6
MOCK48_7
MOCK48_8
MOCK48_9
MOCK48_10
MOCK24_1
MOCK24_2
MOCK24_3
MOCK24_4
MOCK24_5
MOCK24_6
MOCK24_7
MOCK24_8
MOCK24_9
MOCK24_10


## Initialize Reconstruction

In [4]:
## Set Reconstruction Parameters
image_dim = (2048,2448)
wavelength = 525
swing = 0.05
N_channel = 4
NA_obj = 1.1
NA_illu = 0.4
mag = 40
N_slices = 65
z_step = 0.25
pad_z = 0
pixel_size = 6.5
bg_option = 'local_fit'
n_media = 1.33

save_dir = '/gpfs/CompMicro/projects/HAE/2021_02_03_40x_04NA_A549/'


reconstructor = initialize_reconstructor(image_dim, wavelength, swing, N_channel, NA_obj, NA_illu, mag, N_slices, z_step, pad_z, 
                                         pixel_size, bg_option = bg_option, n_media = n_media, use_gpu=False, gpu_id = 0)



Initializing Reconstructor...
Finished Initializing Reconstructor (1.59 min)


## Reconstruct Batch

In [5]:
zarr_store = zarr.open('/gpfs/CompMicro/projects/A549/2021_02_18_40x_04NA_A549/SlideA.zarr')

In [6]:
zarr_store['Mock24']['Phase3D'].info

Name,/Mock24/Phase3D
Type,zarr.core.Array
Data type,float32
Shape,"(10, 65, 2048, 2448)"
Chunk shape,"(1, 65, 2048, 2448)"
Order,C
Read-only,False
Compressor,"Blosc(cname='zstd', clevel=1, shuffle=SHUFFLE, blocksize=0)"
Store type,zarr.storage.DirectoryStore
No. bytes,13035110400 (12.1G)
No. bytes stored,393


In [7]:
bg_path = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_18_40x_04NA_A549/SlideA/BG/'
bg_data = load_bg(bg_path, 2048,2448)

In [ ]:
for pos in range(10,20):
    ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(slideA['Reshape_test'][pos], bg_data, reconstructor, method = "Tikhonov",
                                                                   reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                                   lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)
        
    pos_idx = pos-10    
    zarr_store['RSV24']['BF'][pos_idx] = BF_stack
    zarr_store['RSV24']['Retardance'][pos_idx] = ret_stack
    zarr_store['RSV24']['Orientation'][pos_idx] = ori_stack
    zarr_store['RSV24']['Phase3D'][pos_idx] = np.transpose(phase3D,(2,0,1))

Computing Birefringence...
Finished Computing Birefringence (1.49 min)
Computing 3d Phase...
Finished Reconstruction (2.04 min)

Computing Birefringence...
Finished Computing Birefringence (1.48 min)
Computing 3d Phase...
Finished Reconstruction (2.04 min)

Computing Birefringence...
Finished Computing Birefringence (1.50 min)
Computing 3d Phase...
Finished Reconstruction (2.04 min)



In [ ]:
%%time
for sample in range(len(sample_folders)):

    
    zarr_store = init_zarr_store(save_dir+sample_folders[sample])
    
    for pos in range(len(c1)):
        
        ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(c1[pos], bg, reconstructor, method = "Tikhonov",
                                                               reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                               lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)
        
        
        zarr_store['C1']['BF'][pos] = BF_stack
        zarr_store['C1']['Retardance'][pos] = ret_stack
        zarr_store['C1']['Orientation'][pos] = ori_stack
        zarr_store['C1']['Phase3D'][pos] = np.transpose(phase3D,(2,0,1))
        

    for pos2 in range(len(c2)):
        
        ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(c2[pos2], bg, reconstructor, method = "Tikhonov",
                                                                       reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                                       lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)
        
        zarr_store['C2']['BF'][pos2] = BF_stack
        zarr_store['C2']['Retardance'][pos2] = ret_stack
        zarr_store['C2']['Orientation'][pos2] = ori_stack
        zarr_store['C2']['Phase3D'][pos2] = np.transpose(phase3D,(2,0,1))
        

## Reconstruct Single Position

In [ ]:
%%time
sample = 1 #0-5
pos = 0   #0-4

c1, c2, bg = gather_bg_and_positions(data_paths[sample])

zarr_store = init_zarr_store(save_dir+'ReconOrderTest4')

ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(c1[pos], bg, reconstructor, method = "Tikhonov",
                                                               reg_re = 1e-4, reg_im = 1e-4,rho = 1e-5, 
                                                               lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)

# zarr_store['C1']['BF'][pos] = BF_stack
# zarr_store['C1']['Retardance'][pos] = ret_stack
# zarr_store['C1']['Orientation'][pos] = ori_stack
zarr_store['C1']['Phase3D'][pos] = np.transpose(phase3D,(2,0,1))

In [ ]:
## View Retardance
wo.visual.image_stack_viewer_fast(ret_stack, vrange=(0,20))

In [ ]:
## View Phase
wo.visual.image_stack_viewer_fast(np.transpose(phase3D,(2,0,1)))